In [1]:
pip install selenium pandas openpyxl beautifulsoup4 webdriver-manager

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [openpyxl]1/2 [openpyxl]
Note: you may need to restart the kernel to use updated packages.


In [4]:
import os
import time
import pandas as pd
from io import StringIO
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import Select
from webdriver_manager.chrome import ChromeDriverManager


def scrape_unmsm_results():
    # 1. Definir la ruta absoluta de la carpeta de salida
    output_dir = "/Users/macbookair/Desktop/Data Science con Python/Output Homework 1"
    
    # Nos aseguramos de que la ruta exista por si acaso
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    # 2. Configurar Selenium (Usamos ChromeDriverManager para evitar problemas de versión)
    options = webdriver.ChromeOptions()
    # options.add_argument('--headless') # Descomenta esta línea para que el navegador no se abra visualmente
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    
    url_base = "https://admision.unmsm.edu.pe/Website20262/A/"
    url_principal = f"{url_base}A.html"
    
    todas_las_carreras_df = []

    try:
        print("Accediendo a la página principal...")
        driver.get(url_principal)
        WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.TAG_NAME, "table")))

        # 3. Extraer los links de todas las carreras
        # La UNMSM suele colocar los links dentro de una tabla en la página principal
        elementos_a = driver.find_elements(By.XPATH, "//table//a")
        
        carreras = []
        for a in elementos_a:
            href = a.get_attribute('href')
            nombre_carrera = a.text.strip()
            if href and '.html' in href:
                carreras.append({'nombre': nombre_carrera, 'url': href})
        
        print(f"Se encontraron {len(carreras)} carreras. Iniciando extracción...")

        # 4. Iterar carrera por carrera
        for carrera in carreras:
            print(f"Procesando: {carrera['nombre']}")
            driver.get(carrera['url'])
            
            # Esperar a que la tabla DataTables cargue
            WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.TAG_NAME, "table")))
            time.sleep(1) # Pequeña pausa para asegurar que el JS de DataTables se ejecute
            
            # --- EL TRUCO PARA DATATABLES ---
            # Buscamos el selector de cantidad de registros (suele tener la clase 'form-control' o terminar en '_length')
            try:
                # Intentamos buscar el <select> de DataTables
                select_element = driver.find_element(By.XPATH, "//select[contains(@name, 'length')]")
                select = Select(select_element)
                
                # Seleccionamos la última opción (que suele ser "All" o "-1")
                select.select_by_index(len(select.options) - 1)
                time.sleep(1) # Esperamos a que la tabla se expanda
            except Exception as e:
                # Si falla el selector por interfaz, inyectamos JavaScript para forzar a DataTables a mostrar todo
                print("  [!] Menú no encontrado, forzando expansión vía JavaScript...")
                try:
                    # El ID clásico en las páginas de San Marcos suele ser 'example' u 'oculto'
                    driver.execute_script("var table = $('.dataTable').DataTable(); table.page.len(-1).draw();")
                    time.sleep(1)
                except Exception as js_e:
                    print("  [!] No se pudo forzar JS, se extraerá lo visible.")

            # 5. Extraer el HTML de la página ya expandida
            html_expandido = driver.page_source
            
            # Usar Pandas para leer las tablas del HTML usando StringIO
            tablas = pd.read_html(StringIO(html_expandido))
            
            if tablas:
                # Asumimos que la tabla de resultados es la primera o la más grande
                df_carrera = tablas[0]
                
                # Limpieza básica
                df_carrera = df_carrera.dropna(how='all') # Eliminar filas vacías
                df_carrera['Carrera'] = carrera['nombre'] # Añadir columna con el nombre de la carrera
                
                todas_las_carreras_df.append(df_carrera)
                print(f"  -> Se extrajeron {len(df_carrera)} postulantes.")
            else:
                print(f"  -> No se encontraron tablas en {carrera['nombre']}")

    except Exception as e:
        print(f"Ocurrió un error global: {e}")
    finally:
        driver.quit()

    # 6. Consolidar y exportar a Excel
    if todas_las_carreras_df:
        print("Consolidando información...")
        df_final = pd.concat(todas_las_carreras_df, ignore_index=True)
        
        ruta_salida = os.path.join(output_dir, "resultados_unmsm_2026_consolidado.xlsx")
        df_final.to_excel(ruta_salida, index=False, engine='openpyxl')
        print(f"¡Éxito! Archivo guardado en: {ruta_salida}")
    else:
        print("No se extrajo ningún dato para consolidar.")

# Ejecutar el script
if __name__ == "__main__":
    scrape_unmsm_results()

Accediendo a la página principal...
Se encontraron 111 carreras. Iniciando extracción...
Procesando: ADMINISTRACIÓN
  [!] Menú no encontrado, forzando expansión vía JavaScript...
  -> Se extrajeron 553 postulantes.
Procesando: ADMINISTRACIÓN - CHILCA
  [!] Menú no encontrado, forzando expansión vía JavaScript...
  -> Se extrajeron 33 postulantes.
Procesando: ADMINISTRACIÓN - HUARAL
  [!] Menú no encontrado, forzando expansión vía JavaScript...
  -> Se extrajeron 31 postulantes.
Procesando: ADMINISTRACIÓN - S.J.L
  [!] Menú no encontrado, forzando expansión vía JavaScript...
  -> Se extrajeron 44 postulantes.
Procesando: ADMINISTRACIÓN - VILLA RICA
  [!] Menú no encontrado, forzando expansión vía JavaScript...
  -> Se extrajeron 33 postulantes.
Procesando: ADMINISTRACIÓN DE LA GASTRONOMÍA
  [!] Menú no encontrado, forzando expansión vía JavaScript...
  -> Se extrajeron 127 postulantes.
Procesando: ADMINISTRACIÓN DE NEGOCIOS INTERNACIONALES
  [!] Menú no encontrado, forzando expansión ví